In [208]:
import numpy as np

M = np.array([[-2, -1, 1, 0, -1],
              [-1, 0, 1, 0, 0],
              [1, 1, 0, 0, 1],
              [-1, -1, 0, -1, -1],
              [0, 0, 0, 0, -1]])
A = (M + M.T)
eigenvalues, eigenvectors = np.linalg.eig(A)
D = np.diag(eigenvalues)
P = eigenvectors
P_inv = np.linalg.inv(P)
A_reconstructed = P @ D @ P_inv
print("\nvalidity check:\n", np.max(A_reconstructed - A))
print(eigenvalues)


validity check:
 2.4424906541753444e-15
[-6.79128785e+00 -6.04356314e-16 -2.20871215e+00  2.30277564e+00
 -1.30277564e+00]


In [209]:
def is_dependent_col(matrix, col_idx):
    target_col = matrix[:, [col_idx]]
    other_cols = np.delete(matrix, col_idx, axis=1)
    rank_others = np.linalg.matrix_rank(other_cols)
    rank_full = np.linalg.matrix_rank(matrix)
    return rank_full == rank_others

def is_dependent_row(matrix, row_idx):
    target_row = matrix[[row_idx], :]
    other_rows = np.delete(matrix, row_idx, axis=0)
    rank_others = np.linalg.matrix_rank(other_rows)
    rank_full = np.linalg.matrix_rank(matrix)
    return rank_full == rank_others


for i in range(5):
    print(f"Is col/row 2 dependent? {is_dependent_col(A, i)}/{is_dependent_row(A, 0)}")

Is col/row 2 dependent? True/True
Is col/row 2 dependent? True/True
Is col/row 2 dependent? True/True
Is col/row 2 dependent? False/True
Is col/row 2 dependent? False/True


In [210]:
def Mk(k):
    return np.array([[0,1,0,0],
                     [1,0,0,1],
                     [-1,0,-2*k+1,-2*k+1],
                     [0,0,-2*k+2,-2*k+1]])
def ak(k):
    return -2*k 
def ck(k):
    return np.array([[-1,1,-2*k+2,-2*k+1]])
def bk(k):
    return np.array([[-1,1,-2*k+1,-2*k+2]]).T

In [234]:
def genSeifertMatrix(k):
    M = np.zeros([4*k+1, 4*k+1])
    M[0,0] = ak(k)
    for i in range(k):
        ci = 1+4*i
        M[ci:ci+4, 0:1] = bk(i+1)
        M[0:1, ci:ci+4] = ck(i+1)
    for i in range(k):
        for j in range(k):
            cy, cx = 1+4*i, 1+4*j
            if i > j:
                M[cx:cx+4, cy+2:cy+3] = bk(j+1) 
                M[cx:cx+4, cy+3:cy+4] = bk(j+1) 
            elif i < j:
                M[cx+2:cx+3, cy:cy+4] = ck(i+1) 
                M[cx+3:cx+4, cy:cy+4] = ck(i+1) 
            else:
                M[1+4*i:1+4*(i+1), 1+4*j:1+4*(j+1)] = Mk(i+1)
    return M

M = genSeifertMatrix(2)
A = M+M.T
print(A)

[[-8. -2.  2. -1. -1. -2.  2. -5. -5.]
 [-2.  0.  2. -1.  0.  0.  0. -2. -2.]
 [ 2.  2.  0.  0.  1.  0.  0.  2.  2.]
 [-1. -1.  0. -2. -1.  0.  0. -1. -1.]
 [-1.  0.  1. -1. -2.  0.  0. -1. -1.]
 [-2.  0.  0.  0.  0.  0.  2. -1.  0.]
 [ 2.  0.  0.  0.  0.  2.  0.  0.  1.]
 [-5. -2.  2. -1. -1. -1.  0. -6. -5.]
 [-5. -2.  2. -1. -1.  0.  1. -5. -6.]]


In [228]:
for i in range(1, 10):
    M = genSeifertMatrix(i)
    A=M+M.T
    print(f"Is col/row 0 dependant for k={i}? {is_dependent_col(A, 0)}/{is_dependent_row(A, 0)}")

Is col/row 0 dependant for k=1? True/True
Is col/row 0 dependant for k=2? True/True
Is col/row 0 dependant for k=3? True/True
Is col/row 0 dependant for k=4? True/True
Is col/row 0 dependant for k=5? True/True
Is col/row 0 dependant for k=6? True/True
Is col/row 0 dependant for k=7? True/True
Is col/row 0 dependant for k=8? True/True
Is col/row 0 dependant for k=9? True/True


In [233]:
N = 2
M = genSeifertMatrix(N)
A = M+M.T
for i in range(N):
    for j in range(2):
        A[0, :] -= (-1)**j * A[4*i+1+j, :]
        A[:, 0] -= (-1)**j * A[:, 4*i+1+j]
print(A)

[[ 0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  2. -1.  0.  0.  0. -2. -2.]
 [ 0.  2.  0.  0.  1.  0.  0.  2.  2.]
 [ 0. -1.  0. -2. -1.  0.  0. -1. -1.]
 [ 0.  0.  1. -1. -2.  0.  0. -1. -1.]
 [ 0.  0.  0.  0.  0.  0.  2. -1.  0.]
 [ 0.  0.  0.  0.  0.  2.  0.  0.  1.]
 [ 0. -2.  2. -1. -1. -1.  0. -6. -5.]
 [ 0. -2.  2. -1. -1.  0.  1. -5. -6.]]


In [240]:
N=100
M = genSeifertMatrix(N)
A=M+M.T
A = A[1:,1:]

for i in range(N-1):
    for j in range(i+1, N):
        for k in range(2):
            A[4*j+2+k, :] -= A[4*i, :] - A[4*i+1, :]
            A[:, 4*j+2+k] -= A[:, 4*i] - A[:, 4*i+1]
eigenvalues = []
for i in range(N):
    assert np.array_equal(A[4*i:4*(i+1),4*i:4*(i+1)], A[0:4,0:4])
    e, _  = np.linalg.eig(A[4*i:4*(i+1),4*i:4*(i+1)])
    eigenvalues.append(e)
eigenvalues = np.concatenate(eigenvalues)

pos, neg = np.sum(eigenvalues > 0), np.sum(eigenvalues < 0)
print(f"Positive: {pos}, Negative: {neg}")
print(f"Signature: {pos-neg}")

Positive: 100, Negative: 300
Signature: -200
